In [1]:
##DEPENDENCIAS
!pip install gensim pymongo python-dotenv scikit-learn matplotlib plotly

  Using cached gensim-4.4.0-cp312-cp312-win_amd64.whl.metadata (8.6 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached matplotlib-3.10.8-cp312-cp312-win_amd64.whl.metadata (52 kB)
  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached smart_open-7.5.1-py3-none-any.whl.metadata (24 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp312-cp312-win_amd64.whl.metadata (119 kB)
  Using cached kiwisolver-1.5.0-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.1.1-cp312-cp312-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached wrapt-2.1.2-cp312-cp312-win_amd64.whl.metadata (7.6 kB)
Using

In [4]:
##IMPORTACIONES Y CONEXION A MONGODB
import gensim
from gensim.models import Word2Vec
import pandas as pd
import numpy as np
from pymongo import MongoClient
from dotenv import load_dotenv
import os
import re
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
import plotly.express as px
import nltk

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")
client = MongoClient(MONGO_URI)
db = client["proyecto2_musica"]
coleccion = db["canciones"]

print("Conectado a MongoDB Atlas")
print(f"Total canciones: {coleccion.count_documents({})}")

Conectado a MongoDB Atlas
Total canciones: 10133


In [5]:
##CARGAR CORPUS DESDE MONGODB
print("Cargando corpus desde MongoDB...")

canciones = list(coleccion.find(
    {"letra": {"$exists": True, "$ne": None}},
    {"letra": 1, "genero": 1, "artista": 1, "titulo": 1, "idioma": 1, "_id": 0}
))

df = pd.DataFrame(canciones)
print(f"{len(df)} canciones cargadas")
print(f"\nPor genero:")
print(df["genero"].value_counts())

Cargando corpus desde MongoDB...
10134 canciones cargadas

Por genero:
genero
pop        2494
country    1950
blues      1610
rock       1404
jazz       1328
reggae      893
hip hop     321
Name: count, dtype: int64


In [6]:
##PROCESAR DATOS
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_es = set(stopwords.words("spanish"))
stop_en = set(stopwords.words("english"))
stopwords_all = stop_es.union(stop_en)

def limpiar_texto(texto):
    if not isinstance(texto, str):
        return []
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúüñ\s]", "", texto)
    tokens = texto.split()
    tokens = [t for t in tokens if t not in stopwords_all and len(t) > 2]
    return tokens

df["tokens"] = df["letra"].apply(limpiar_texto)
df = df[df["tokens"].apply(len) > 10]

print(f"Preprocesamiento completo")
print(f"Canciones validas: {len(df)}")
print(f"Ejemplo tokens: {df['tokens'].iloc[0][:10]}")

Preprocesamiento completo
Canciones validas: 10062
Ejemplo tokens: ['mind', 'fade', 'pink', 'cashmere', 'summer', 'nights', 'summer', 'nights', 'summer', 'nights']


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\98248\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
##ENTRENAR A CBOW Y SKIP GRAM
corpus = df["tokens"].tolist()

print("Entrenando modelo CBOW...")
modelo_cbow = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=3,
    sg=0,
    workers=4,
    epochs=10
)

print("Entrenando modelo Skip-Gram...")
modelo_skipgram = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,
    workers=4,
    epochs=10
)

print(f"Modelos entrenados")
print(f"Vocabulario CBOW: {len(modelo_cbow.wv)} palabras")
print(f"Vocabulario Skip-Gram: {len(modelo_skipgram.wv)} palabras")

modelo_cbow.save("../data/processed/w2v_cbow.model")
modelo_skipgram.save("../data/processed/w2v_skipgram.model")
print("Modelos guardados")

Entrenando modelo CBOW...
Entrenando modelo Skip-Gram...
Modelos entrenados
Vocabulario CBOW: 12941 palabras
Vocabulario Skip-Gram: 12941 palabras
Modelos guardados


In [8]:
##CAMPOS SEMANTICOS POR GENERO
palabras_clave = {
    "pop": ["love", "heart", "baby", "feel"],
    "rock": ["fire", "fight", "live", "free"],
    "hip hop": ["money", "street", "real", "life"],
    "country": ["home", "road", "truck", "beer"],
    "jazz": ["blues", "soul", "night", "sweet"],
    "reggae": ["jah", "rasta", "peace", "fire"],
}

print("Campos semanticos por genero (CBOW):\n")
for genero, palabras in palabras_clave.items():
    print(f"{genero.upper()}:")
    for palabra in palabras:
        if palabra in modelo_cbow.wv:
            similares = modelo_cbow.wv.most_similar(palabra, topn=5)
            print(f"  '{palabra}' -> {[w for w,s in similares]}")
    print()

Campos semanticos por genero (CBOW):

POP:
  'love' -> ['naturally', 'bobby', 'fluently', 'sweetest', 'overjoy']
  'heart' -> ['mend', 'apart', 'part', 'ache', 'hearts']
  'baby' -> ['hunnid', 'born', 'woooooh', 'yeeeeh', 'bitsy']
  'feel' -> ['felt', 'ashamed', 'heartbeat', 'know', 'heal']

ROCK:
  'fire' -> ['pruitt', 'cedric', 'bullets', 'overturn', 'confetti']
  'fight' -> ['fuss', 'battle', 'ohah', 'authority', 'cuss']
  'live' -> ['life', 'frontlines', 'applauseplause', 'eternity', 'idolize']
  'free' -> ['unconditionally', 'unchain', 'doubtful', 'thinkers', 'gainst']

HIP HOP:
  'money' -> ['banker', 'bank', 'cept', 'canister', 'dollar']
  'street' -> ['meet', 'middle', 'woods', 'deadend', 'standin']
  'real' -> ['appeal', 'silverlake', 'jawbreaker', 'deal', 'smooth']
  'life' -> ['live', 'strife', 'chapters', 'short', 'last']

COUNTRY:
  'home' -> ['roam', 'graver', 'someplace', 'snowwhite', 'wyoming']
  'road' -> ['travel', 'runner', 'bumpy', 'highway', 'graveyard']
  'truck' 

In [11]:
##ANALOGIAS VECTORIALES
print("Analogias vectoriales:\n")

analogias = [
    (["love", "woman"], ["man"]),
    (["night", "moon"], ["day"]),
    (["happy", "laugh"], ["cry"]),
    (["rock", "loud"], ["soft"]),
    (["country", "road"], ["city"]),
]

for positivos, negativos in analogias:
    try:
        resultado = modelo_cbow.wv.most_similar(
            positive=positivos,
            negative=negativos,
            topn=3
        )
        print(f"  {positivos} - {negativos} = {[w for w,s in resultado]}")
    except KeyError as e:
        print(f"  Palabra no encontrada: {e}")

Analogias vectoriales:

  ['love', 'woman'] - ['man'] = ['misery', 'heartaches', 'heart']
  ['night', 'moon'] - ['day'] = ['daylight', 'dark', 'light']
  ['happy', 'laugh'] - ['cry'] = ['dutch', 'aint', 'bubbly']
  ['rock', 'loud'] - ['soft'] = ['crew', 'prop', 'holler']
  ['country', 'road'] - ['city'] = ['roads', 'travel', 'crescendo']


In [12]:
##SIMILITUD ENTRE GENEROS
def vector_promedio_genero(genero, modelo):
    canciones_genero = df[df["genero"] == genero]["tokens"].tolist()
    vectores = []
    for tokens in canciones_genero:
        for token in tokens:
            if token in modelo.wv:
                vectores.append(modelo.wv[token])
    if vectores:
        return np.mean(vectores, axis=0)
    return None

generos = df["genero"].dropna().unique()
vectores_genero = {}

for genero in generos:
    vec = vector_promedio_genero(genero, modelo_cbow)
    if vec is not None:
        vectores_genero[genero] = vec

print("Similitud coseno entre generos:\n")
generos_lista = list(vectores_genero.keys())
matriz = np.array([vectores_genero[g] for g in generos_lista])
similitudes = cosine_similarity(matriz)

df_sim = pd.DataFrame(similitudes, index=generos_lista, columns=generos_lista)
print(df_sim.round(3))

Similitud coseno entre generos:

          rock    pop   jazz  country  blues  reggae  hip hop
rock     1.000  0.991  0.993    0.985  0.992   0.981    0.878
pop      0.991  1.000  0.994    0.991  0.994   0.980    0.901
jazz     0.993  0.994  1.000    0.989  0.993   0.984    0.889
country  0.985  0.991  0.989    1.000  0.995   0.963    0.854
blues    0.992  0.994  0.993    0.995  1.000   0.977    0.879
reggae   0.981  0.980  0.984    0.963  0.977   1.000    0.930
hip hop  0.878  0.901  0.889    0.854  0.879   0.930    1.000


In [13]:
##VISUALIZACIONES Y TSNE
palabras_freq = Counter([token for tokens in corpus for token in tokens])
top_palabras = [p for p, _ in palabras_freq.most_common(200) if p in modelo_cbow.wv]

vectores = np.array([modelo_cbow.wv[p] for p in top_palabras])

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
vectores_2d = tsne.fit_transform(vectores)

fig = px.scatter(
    x=vectores_2d[:, 0],
    y=vectores_2d[:, 1],
    text=top_palabras,
    title="t-SNE - Top 200 palabras Word2Vec",
    width=900,
    height=600
)
fig.update_traces(textposition="top center", marker=dict(size=5))
fig.show()

In [14]:
##CERRAR CONEXIONES
client.close()
print("Conexion cerrada")

Conexion cerrada
